In [ ]:
import torch
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from lorentz import Lorentz

# ── config ────────────────────────────────────────────────────────────────────
CKPT   = 'ckpt/final_my_graph.ckpt'
VOCAB  = 'enoa_vocab.pkl'
COLLAB = 'code/dataset/Original/global/global-genre_network-2019.csv'
DIM    = 2
# ─────────────────────────────────────────────────────────────────────────────

genres = pickle.load(open(VOCAB, 'rb'))
n_items = len(genres)
genre_to_idx = {g: i for i, g in enumerate(genres)}

net = Lorentz(n_items, DIM + 1)
net.load_state_dict(torch.load(CKPT, map_location='cpu'))
net.eval()

# get full lorentz embeddings (index 0 is padding, genre i is at index i+1)
lorentz_table = net.table.weight.data.cpu().numpy()

print(f'Loaded {n_items} genres')
print(f'Lorentz table shape: {lorentz_table.shape}')

In [ ]:
def lorentz_distance(u, v):
    """
    Lorentz (hyperbolic) distance between two points on the hyperboloid.
    d(u,v) = arcosh(-<u,v>_L)
    Minkowski inner product: <u,v>_L = -u[0]*v[0] + sum(u[1:]*v[1:])
    """
    inner = -u[0]*v[0] + np.dot(u[1:], v[1:])
    inner = np.clip(-inner, 1 + 1e-6, None)
    return float(np.arccosh(inner))

def get_embedding(genre_name):
    if genre_name not in genre_to_idx:
        return None
    idx = genre_to_idx[genre_name]
    return lorentz_table[idx + 1]  # +1 because index 0 is padding

# sanity check
for g1, g2 in [('pop', 'dance pop'), ('pop', 'rap'), ('rap', 'reggaeton'), ('rock', 'metal')]:
    u, v = get_embedding(g1), get_embedding(g2)
    if u is not None and v is not None:
        print(f'  {g1} <-> {g2}: {lorentz_distance(u, v):.4f}')

In [ ]:
# load collaboration network
collab = pd.read_csv(COLLAB, sep='\t')
collab = collab[collab['source'] != collab['target']]
print(f'Collaboration edges: {len(collab)}')
print(collab.head())

In [ ]:
# compute lorentz distance for each collaboration pair
rows = []
skipped = 0

for _, row in collab.iterrows():
    u = get_embedding(row['source'])
    v = get_embedding(row['target'])
    if u is None or v is None:
        skipped += 1
        continue
    rows.append({
        'source':      row['source'],
        'target':      row['target'],
        'weight':      row['weight'],
        'avg_streams': row['avg_streams'],
        'distance':    lorentz_distance(u, v)
    })

df = pd.DataFrame(rows)
print(f'Computed distances for {len(df)} pairs')
print(f'Skipped {skipped} pairs (not in ENOA vocab)')
print()
print(df[['distance', 'avg_streams']].describe().round(2))

In [ ]:
# ── Main analysis: correlation between distance and popularity ────────────────
r, p_r   = stats.pearsonr(df['distance'], df['avg_streams'])
rho, p_s = stats.spearmanr(df['distance'], df['avg_streams'])

print(f'Pearson  r   = {r:.4f}  (p = {p_r:.4f})')
print(f'Spearman rho = {rho:.4f}  (p = {p_s:.4f})')
print()
if p_r < 0.05:
    direction = 'positive' if r > 0 else 'negative'
    print(f'Significant {direction} Pearson correlation!')
    if r > 0:
        print('-> More diverse collaborations (larger distance) tend to be more popular')
    else:
        print('-> More similar collaborations (smaller distance) tend to be more popular')
else:
    print('No significant Pearson correlation (p >= 0.05)')

if p_s < 0.05:
    direction = 'positive' if rho > 0 else 'negative'
    print(f'Significant {direction} Spearman correlation!')
else:
    print('No significant Spearman correlation (p >= 0.05)')

In [ ]:
# ── Scatter plots ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['distance'], df['avg_streams'], alpha=0.4, s=15, color='steelblue')
axes[0].set_xlabel('Lorentz Distance')
axes[0].set_ylabel('Average Streams')
axes[0].set_title('Distance vs Popularity (raw)')

log_streams = np.log10(df['avg_streams'] + 1)
axes[1].scatter(df['distance'], log_streams, alpha=0.4, s=15, color='tomato')
z = np.polyfit(df['distance'], log_streams, 1)
x_line = np.linspace(df['distance'].min(), df['distance'].max(), 100)
axes[1].plot(x_line, np.poly1d(z)(x_line), color='black', linewidth=2, label='trend')
axes[1].set_xlabel('Lorentz Distance')
axes[1].set_ylabel('log10(Average Streams)')
axes[1].set_title('Distance vs Popularity (log scale)')
axes[1].legend()

plt.suptitle(f'Pearson r={r:.3f} (p={p_r:.3f})  |  Spearman rho={rho:.3f} (p={p_s:.3f})', fontsize=11)
plt.tight_layout()
plt.savefig('distance_vs_popularity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to distance_vs_popularity.png')

In [ ]:
# ── Quartile analysis ─────────────────────────────────────────────────────────
df['distance_quartile'] = pd.qcut(df['distance'], q=4,
                                   labels=['Q1 (closest)', 'Q2', 'Q3', 'Q4 (furthest)'])

stats_q = df.groupby('distance_quartile', observed=True)['avg_streams'].agg(
    mean='mean', median='median', count='count'
).round(0)
print('Streams by distance quartile:')
print(stats_q)

fig, ax = plt.subplots(figsize=(8, 5))
stats_q['mean'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Distance Quartile')
ax.set_ylabel('Mean Average Streams')
ax.set_title('Mean Popularity by Genre Distance Quartile')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('quartile_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to quartile_analysis.png')

In [ ]:
# ── Top examples ──────────────────────────────────────────────────────────────
print('=== Most popular collaborations ===')
print(df.nlargest(10, 'avg_streams')[['source','target','distance','avg_streams']].to_string(index=False))

print()
print('=== Most diverse collaborations (largest distance) ===')
print(df.nlargest(10, 'distance')[['source','target','distance','avg_streams']].to_string(index=False))

print()
print('=== Most similar collaborations (smallest distance) ===')
print(df.nsmallest(10, 'distance')[['source','target','distance','avg_streams']].to_string(index=False))